# archlux in 10 minutes

A generated floor plan is close to right but not quite: rooms overlap, a load-bearing
wall is crossed, a room is too small. `archlux` repairs it with the **smallest** wall
moves and proves the result valid, exactly. This notebook walks through the whole
loop on a shipped example: read, check, repair under the French regulation, report,
draw, export.

Run it from the `examples/` directory, after `pip install archlux`.

In [ ]:
from pathlib import Path

import archlux as ax

here = Path.cwd()
plan = ax.Plan.from_json(here / "t3_load_bearing.json")
for room in plan.rooms:
    print(f"{room.id:10} {room.type:12} {room.w:.2f} x {room.h:.2f} m = {room.area:.2f} m²")

## 1. Choose the regulation

A preset gives sourced thresholds, and says what it cannot check. Here, new
collective housing in France.

In [ ]:
fr = ax.preset("fr/new-collective-housing")
for source in fr.sources:
    print(f"{source.rule:22} {source.text}")
print()
print("Not checked:", *fr.not_checked, sep="\n  - ")

## 2. Describe the context

The load-bearing walls are the structure the repair may not move; the outline is the
site; the orientation is the azimuth of the plan.

The repair warns about room types the regulation leaves without a minimum area (here
the bathroom): it is told, not silently ignored.

In [ ]:
load_bearing = tuple(wall for wall in plan.walls if wall.load_bearing)
ctx = ax.Context(
    ax.Structure(load_bearing), ax.Orientation(0.0), fr.regulation, outline=plan.outline
)
print(len(load_bearing), "load-bearing wall(s)")

## 3. Repair

`legalize` returns the repaired plan with its certificate. `tiling=True` also closes
the gaps, so the rooms tile the outline.

In [ ]:
fixed = ax.legalize(plan, ctx, tiling=True)
assert fixed.certificate is not None
print("valid:", fixed.certificate.geometry.valid)
print(f"largest move: {fixed.certificate.geometry.max_displacement:.2f} m")

## 4. Read the report

Room by room: size before and after, minimum, margin, how far the walls moved, and
what the regulation does not check. `lang="fr"` gives it in French.

In [ ]:
print(fixed.repair_report(plan, fr.regulation, lang="fr", not_checked=fr.not_checked))

## 5. Draw before and after

Same scale on both sides, so a moved wall looks moved. In Jupyter the drawing shows
inline; elsewhere it is written to `compare.svg`.

In [ ]:
from archlux.export import compare_svg

drawing = compare_svg(plan, fixed, outline=plan.outline, walls=fixed.walls)
Path("compare.svg").write_text(drawing, encoding="utf-8")
try:
    from IPython.display import SVG, display

    display(SVG(drawing))
except ImportError:
    print("wrote compare.svg")

## 6. Save and export

JSON keeps the certificate; DXF and IFC go to CAD and BIM programs.

In [ ]:
fixed.to_json("fixed.json")
fixed.to_dxf("fixed.dxf")
export = fixed.to_ifc("fixed.ifc")
print("IFC written:", export)

## Next

- The same from a terminal: `archlux legalize t3_load_bearing.json --preset fr/new-collective-housing --tiling --report --lang fr`.
- Start from your own drawing: `ax.Plan.from_dxf("plan.dxf")` (`pip install "archlux[cad]"`).
- Feasibility before repair: `ax.feasibility.is_feasible(plan, ctx)`.